In [51]:
import pandas as pd

## 1. Cargar los datasets

In [52]:
master_team = pd.read_parquet(
    r"Datasets\Team Stats - Kaggle\nba_master_team_v1.parquet"
)

In [53]:
team_summary = pd.read_csv(
    r"Datasets\Team Stats - Kaggle\Team Summaries.csv"
)

team_pg = pd.read_csv(
    r"Datasets\Team Stats - Kaggle\Team Stats Per Game.csv"
)

team_100 = pd.read_csv(
    r"Datasets\Team Stats - Kaggle\Team Stats Per 100 Poss.csv"
)

## 2. Limpieza Team Stats

#### Filtramos temporadas

In [54]:
team_summary = team_summary[
    team_summary["season"].between(2001, 2025)
].copy()

team_pg = team_pg[
    team_pg["season"].between(2001, 2025)
].copy()

team_100 = team_100[
    team_100["season"].between(2001, 2025)
].copy()

#### Crear columna Season

In [55]:
def season_format(year):

    return f"{year-1}-{str(year)[-2:]}"

In [56]:
for df in [team_summary, team_pg, team_100]:

    df["Season"] = (
        df["season"]
        .apply(season_format)
    )

#### Renombrar Team

In [57]:
for df in [team_summary, team_pg, team_100]:
    df.rename(
        columns={"abbreviation": "Team"},
        inplace=True
    )

#### Eliminar League Average

In [58]:
team_summary = team_summary[
    team_summary["Team"].notna()
].copy()

team_pg = team_pg[
    team_pg["Team"].notna()
].copy()

team_100 = team_100[
    team_100["Team"].notna()
].copy()

#### Auditoría

In [59]:
print(team_summary.shape)
print(team_pg.shape)
print(team_100.shape)

print(
    team_summary[["Season", "Team"]].duplicated().sum()
)

print(
    team_pg[["Season", "Team"]].duplicated().sum()
)

print(
    team_100[["Season", "Team"]].duplicated().sum()
)

(746, 32)
(746, 29)
(746, 29)
0
0
0


## 3. Crear Franchise

In [60]:
FRANCHISE_MAP = {

    "SEA":"OKC",
    "OKC":"OKC",

    "NJN":"BRK",
    "BRK":"BRK",

    "NOH":"NOP",
    "NOK":"NOP",
    "NOP":"NOP",

    "CHH":"CHA",
    "CHO":"CHA",
    "CHA":"CHA",

    "VAN":"MEM",
    "MEM":"MEM"

}

In [61]:
# No modificamos team, añadimos una variable nueva
for df in [

    master_team,

    team_summary,

    team_pg,

    team_100

]:

    df["Franchise"] = (

        df["Team"]

        .replace(FRANCHISE_MAP)

    )

## 4. Seleccionar las variables útiles / Prescindir de las columnas innecesarias

#### Team summary

In [62]:
team_summary_clean = team_summary[
    [
        "Season",
        "Team",
        "Franchise",

        "playoffs",

        "w",
        "l",

        "srs",

        "o_rtg",
        "d_rtg",
        "n_rtg",

        "pace",

        "ts_percent",
        "e_fg_percent",

        "tov_percent",

        "orb_percent",

        "drb_percent"
    ]
].copy()

#### Team Per Game

In [63]:
team_pg_clean = team_pg[
    [
        "Season",
        "Team",
        "Franchise",

        "pts_per_game",

        "ast_per_game",

        "trb_per_game",

        "orb_per_game",

        "drb_per_game",

        "fg_per_game",

        "fga_per_game",

        "x3p_per_game",

        "x3pa_per_game",

        "x3p_percent"
    ]
].copy()

#### Team Per100

In [64]:
team_100_clean = team_100[
    [
        "Season",
        "Team",
        "Franchise",

        "pts_per_100_poss",

        "ast_per_100_poss",

        "tov_per_100_poss",

        "x3p_per_100_poss",

        "x3pa_per_100_poss"
    ]
].copy()

## 5. Construir Team Stats

In [65]:
team_stats = team_summary_clean.merge(
    team_pg_clean,
    on=["Season", "Team", "Franchise"],
    how="left"
)

team_stats = team_stats.merge(
    team_100_clean,
    on=["Season", "Team", "Franchise"],
    how="left"
)

#### Comprobación

In [66]:
print(team_stats.shape)

print(
    team_stats[["Season", "Team"]].duplicated().sum()
)

print(team_stats.isna().sum())

(746, 31)
0
Season               0
Team                 0
Franchise            0
playoffs             0
w                    0
l                    0
srs                  0
o_rtg                0
d_rtg                0
n_rtg                0
pace                 0
ts_percent           0
e_fg_percent         0
tov_percent          0
orb_percent          0
drb_percent          0
pts_per_game         0
ast_per_game         0
trb_per_game         0
orb_per_game         0
drb_per_game         0
fg_per_game          0
fga_per_game         0
x3p_per_game         0
x3pa_per_game        0
x3p_percent          0
pts_per_100_poss     0
ast_per_100_poss     0
tov_per_100_poss     0
x3p_per_100_poss     0
x3pa_per_100_poss    0
dtype: int64


## 6. Renombrar las columnas

In [67]:
team_stats.rename(
    columns={
        "playoffs": "team_playoffs",
        "w": "team_w",
        "l": "team_l",
        "srs": "team_srs",
        "o_rtg": "team_o_rtg",
        "d_rtg": "team_d_rtg",
        "n_rtg": "team_n_rtg",
        "pace": "team_pace",
        "ts_percent": "team_ts_percent",
        "e_fg_percent": "team_e_fg_percent",
        "tov_percent": "team_tov_percent",
        "orb_percent": "team_orb_percent",
        "drb_percent": "team_drb_percent",
        "pts_per_game": "team_pts_pg",
        "ast_per_game": "team_ast_pg",
        "trb_per_game": "team_trb_pg",
        "orb_per_game": "team_orb_pg",
        "drb_per_game": "team_drb_pg",
        "fg_per_game": "team_fg_pg",
        "fga_per_game": "team_fga_pg",
        "x3p_per_game": "team_3p_pg",
        "x3pa_per_game": "team_3pa_pg",
        "x3p_percent": "team_3p_pct",
        "pts_per_100_poss": "team_pts_100",
        "ast_per_100_poss": "team_ast_100",
        "tov_per_100_poss": "team_tov_100",
        "x3p_per_100_poss": "team_3p_100",
        "x3pa_per_100_poss": "team_3pa_100"
    },
    inplace=True
)

## 7. Merge con master_team

In [68]:
master_team_v2 = master_team.merge(
    team_stats,
    on=["Season", "Team", "Franchise"],
    how="left"
)

## 8. Auditoría final

In [69]:
print(master_team_v2.shape)

print(
    master_team_v2[
        ["Player", "Age", "Season"]
    ].duplicated().sum()
)

print(master_team_v2.isna().sum().sort_values(ascending=False).head(20))

(12227, 147)
0
Unnamed: 30_level_0_Awards    10734
Awards_adv                    10734
Awards                        10734
Awards_tot                    10734
% of FG Ast'd_3P               2933
Corner 3s_3P%                  2820
3P%_tot                        1461
Corner 3s_%3PA                 1461
3P%                            1461
FG% by Distance_3P             1461
FG% by Distance_16-3P          1010
FG% by Distance_10-16           834
FT%                             488
FT%_tot                         488
FG% by Distance_3-10            480
FG% by Distance_0-3             322
% of FG Ast'd_2P                263
2P%                              98
FG% by Distance_2P               98
2P%_tot                          98
dtype: int64


In [70]:
print(master_team_v2["team_w"].isna().sum())
print(master_team_v2["team_o_rtg"].isna().sum())
print(master_team_v2["team_3pa_pg"].isna().sum())

0
0
0


## 9. Guardar el dataset

In [71]:
master_team_v2.to_csv(
    r"Datasets\Team Stats - Kaggle\nba_master_team_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

master_team_v2.to_parquet(
    r"Datasets\Team Stats - Kaggle\nba_master_team_v2.parquet",
    index=False
)